# Story Generator · DeepSeek（本地 / Google Colab）

故事分为 **起因、发展、转折、高潮、结局** 五个 section。每个 section 生成三个指定故事日。每一块都按照“检查与预览 → 生成候选 → 人工确认”运行；只有确认后的文本才会进入下一块的 Generation Context。

## 0. 环境初始化

从 GitHub 在 Google Colab 中打开后，**先点击工具栏中的“复制到云端硬盘”**，然后在新打开的个人 Drive 副本里继续操作。这样 Notebook 本身会保存在自己的 Google Drive 中。

在 Drive 副本中依次运行下面三个单元格。首次运行时，Colab 会请求挂载 Google Drive，并把整个公开 GitHub 仓库克隆到 `MyDrive/story-generator-workshop/`；以后会直接复用该云端目录，不会自动覆盖或拉取更新。你可以修改其中的 `src/` 代码和 `configs/` 配置，修改与 `runs/` 生成结果都会保留在自己的 Drive 中。依赖安装可能需要约一分钟。

运行第三个单元格时，先在 DeepSeek 开放平台创建 API Key，再把它复制粘贴到 Colab 出现的输入框中并按回车。输入内容不会显示，也不会写入 Notebook 或 GitHub；只在本次 Colab 运行期间保存在内存中。**本地运行则不会弹出输入框，而是读取项目根目录的 `.env`。**

**不要选择“全部运行”**：每个 section 都需要先检查候选、满意后再确认；全部运行会连续调用 API。

In [ ]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/666feiyu666/story-generator.git"
REPO_REF = "main"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/story-generator-workshop")
    if PROJECT_ROOT.exists() and not (PROJECT_ROOT / ".git").is_dir():
        raise FileExistsError(
            f"{PROJECT_ROOT} 已存在但不是 Git 仓库；请将该目录改名后重试。"
        )
    if not (PROJECT_ROOT / ".git").is_dir():
        print("首次运行：正在把项目复制到你的 Google Drive……")
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", REPO_REF,
                REPO_URL, str(PROJECT_ROOT),
            ],
            check=True,
        )
    else:
        print("正在使用 Google Drive 中已有的项目；不会覆盖你的修改。")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "--requirement", str(PROJECT_ROOT / "requirements-colab.txt"),
        ],
        check=True,
    )
else:
    candidates = (Path.cwd(), Path.cwd() / "story-generator", Path.cwd().parent)
    PROJECT_ROOT = next(
        (
            path.resolve()
            for path in candidates
            if (path / "src" / "story_generator").is_dir()
        ),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "找不到 src/story_generator。请从 story-generator 目录启动 JupyterLab。"
        )

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from importlib.metadata import version

print("Runtime:", "Google Colab" if IN_COLAB else "Local Jupyter")
print("Python:", sys.version.split()[0])
print("langchain-deepseek:", version("langchain-deepseek"))
print("Project root:", PROJECT_ROOT)

In [ ]:
required_paths = (
    PROJECT_ROOT / "src" / "story_generator",
    PROJECT_ROOT / "configs" / "fox_and_crow",
)
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("缺少项目文件：" + "、".join(missing_paths))
print("项目文件检查通过。")

In [ ]:
import getpass
import os
from dotenv import load_dotenv

if IN_COLAB:
    api_key = getpass.getpass(
        "请粘贴刚创建的 DeepSeek API Key（输入不会显示），然后按回车："
    ).strip()
else:
    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
if not api_key:
    source = "Colab 输入框" if IN_COLAB else str(PROJECT_ROOT / ".env")
    raise RuntimeError(f"未从 {source} 获得 DEEPSEEK_API_KEY。")
os.environ["DEEPSEEK_API_KEY"] = api_key
print("DeepSeek API Key 已载入本次运行环境；Key 本身不会显示。")

In [ ]:
from IPython.display import Markdown, display
from story_generator import (
    ModelConfig,
    accept_section,
    build_prompt,
    export_result,
    generate_next_section,
    generate_story,
    load_story_config,
    result_from_context,
)

CONFIG_DIR = PROJECT_ROOT / "configs" / "fox_and_crow"
story_config = load_story_config(CONFIG_DIR)
model_config = ModelConfig(
    model="deepseek-chat",
    temperature=0.7,
    max_tokens=3600,
    timeout=90.0,
    max_retries=2,
)

if "working_context" not in globals():
    working_context = story_config.generation_context
    accepted_usage = {}

print("Configured sections:", story_config.plot_structure.generation_plan())
print("Accepted sections:", [s.label for s in working_context.generated_sections])

In [ ]:
SECTION_ORDER = ("cause", "development", "turning_point", "climax", "resolution")
SECTION_LABELS = {
    "cause": "起因",
    "development": "发展",
    "turning_point": "转折",
    "climax": "高潮",
    "resolution": "结局",
}

def require_section(stage_name):
    label = SECTION_LABELS[stage_name]
    stage = story_config.plot_structure.stage(stage_name)
    if not stage.configured:
        path = CONFIG_DIR / "plot_structure" / f"{stage_name}.json"
        raise RuntimeError(f"当前 section「{label}」尚未配置：{path}")

    accepted = {section.plot_stage for section in working_context.generated_sections}
    if stage_name in accepted:
        raise RuntimeError(f"section「{label}」已经确认，无需再次运行。")

    stage_index = SECTION_ORDER.index(stage_name)
    for previous_name in SECTION_ORDER[:stage_index]:
        previous_label = SECTION_LABELS[previous_name]
        previous = story_config.plot_structure.stage(previous_name)
        if not previous.configured:
            raise RuntimeError(f"前一 section「{previous_label}」尚未配置。")
        if previous_name not in accepted:
            raise RuntimeError(
                f"缺少前一 section「{previous_label}」的已确认文本；请先生成并确认它。"
            )

    plan = story_config.plot_structure.generation_plan()
    position = working_context.current_position.sequence_index
    if position >= len(plan) or plan[position] != stage_name:
        raise RuntimeError(f"当前 Generation Context 尚未到达 section「{label}」。")

def preview_section(stage_name):
    require_section(stage_name)
    prompt = build_prompt(story_config, working_context)
    print("SYSTEM PROMPT\n" + prompt.system)
    print("\nUSER PROMPT\n" + prompt.user)
    return prompt

def generate_candidate(stage_name):
    require_section(stage_name)
    draft = generate_next_section(story_config, working_context, model_config)
    display(Markdown(f"## {draft.section.label}\n\n{draft.section.text}"))
    print("\nContext update candidate:")
    display(draft.context_update.to_dict())
    return draft

def confirm_section(stage_name, draft):
    require_section(stage_name)
    label = SECTION_LABELS[stage_name]
    if draft is None:
        raise RuntimeError(f"请先生成 section「{label}」的候选文本。")
    if draft.section.plot_stage != stage_name:
        raise RuntimeError(f"当前候选文本不属于 section「{label}」。")
    context = accept_section(story_config, working_context, draft)
    for key, value in draft.usage.items():
        if isinstance(value, int) and not isinstance(value, bool):
            accepted_usage[key] = accepted_usage.get(key, 0) + value
    print(f"已确认 section「{label}」。")
    print("Accepted sections:", [s.label for s in context.generated_sections])
    return context

如需从头开始，运行：

```python
working_context = story_config.generation_context
accepted_usage = {}
```

---
## 1. 起因 · Day 1 / Day 2 / Day 3

配置文件：`configs/fox_and_crow/plot_structure/cause.json`

In [ ]:
cause_prompt = preview_section("cause")

In [ ]:
cause_draft = generate_candidate("cause")

In [ ]:
working_context = confirm_section("cause", globals().get("cause_draft"))

---
## 2. 发展 · Day 8 / Day 11 / Day 13

配置文件：`configs/fox_and_crow/plot_structure/development.json`  
前置条件：起因已经生成并确认。

In [ ]:
development_prompt = preview_section("development")

In [ ]:
development_draft = generate_candidate("development")

In [ ]:
working_context = confirm_section("development", globals().get("development_draft"))

---
## 3. 转折 · Day 16 / Day 17 / Day 18

配置文件：`configs/fox_and_crow/plot_structure/turning_point.json`  
前置条件：起因、发展已经生成并确认。

In [ ]:
turning_point_prompt = preview_section("turning_point")

In [ ]:
turning_point_draft = generate_candidate("turning_point")

In [ ]:
working_context = confirm_section("turning_point", globals().get("turning_point_draft"))

---
## 4. 高潮 · Day 30 / Day 31 / Day 32

配置文件：`configs/fox_and_crow/plot_structure/climax.json`  
前置条件：起因、发展、转折已经生成并确认。

In [ ]:
climax_prompt = preview_section("climax")

In [ ]:
climax_draft = generate_candidate("climax")

In [ ]:
working_context = confirm_section("climax", globals().get("climax_draft"))

---
## 5. 结局 · Day 33 / Day 34 / Day 35

配置文件：`configs/fox_and_crow/plot_structure/resolution.json`  
前置条件：起因、发展、转折、高潮已经生成并确认。

In [ ]:
resolution_prompt = preview_section("resolution")

In [ ]:
resolution_draft = generate_candidate("resolution")

In [ ]:
working_context = confirm_section("resolution", globals().get("resolution_draft"))

---
## 6. 查看并导出当前已确认文本

In [ ]:
result = result_from_context(
    story_config,
    working_context,
    model_config,
    accepted_usage,
)
display(Markdown(result.story or "尚无已确认文本。"))
markdown_path, json_path = export_result(result, PROJECT_ROOT / "runs")
print(markdown_path)
print(json_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(markdown_path))
    files.download(str(json_path))

### 可选：一次生成全部五段

配置稳定后可以运行 `full_result = generate_story(story_config, model_config)`。该入口会跳过人工确认，自动接受全部五段。